# 18 · Rétropropagation — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch18_backprop/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 18.Q1 à 18.Q12, 18.R1 à 18.R3, 18.1 à 18.5, 18.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 18.12 à 18.17 | les losses et leur gradient, la méthode très lente, la passe arrière d'une couche, gradient_check, une chasse aux gradients faux (*prochaine session de génération*) | | | |
| B | 18.18 à 18.21 | mlp_backward, le réseau 2-4-4-1 à 37 poids sur les lunes, la même expérience avec scikit-learn, les figures du learning rate (*prochaine session de génération*) | | | |
| C | 18.22 à 18.25 | lire des courbes de loss, balayer le learning rate, des neurones saturés et morts sur MNIST, un test pytest de gradient (*prochaine session de génération*) | | | |
| D | 18.26 à 18.29 | le micro-autograd Value, quatre gradients comparés, MiniMLP, le défi MNIST (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer à la main les deltas, les gradients et un pas de mise à jour d'un petit réseau, avec et sans activation.
- Écrire et vérifier contre PyTorch les fonctions de `mylearn.nn.backward` (les losses avec leur gradient, `dense_backward`, `mlp_backward`, `gradient_check`) et le micro-autograd `mylearn.autograd_mini` (`Value`, `MiniMLP`).
- Entraîner un MLP from scratch et diagnostiquer le learning rate, les plateaux, la saturation et les neurones morts.

**Rappel express.** $\delta_j = \partial L / \partial z_j$ ; sortie, loss quadratique : $\delta_k = (\hat{y}_k - y_k)\,f'(z_k)$ ; caché : $\delta_j = f'(z_j) \sum_k w_{jk}\,\delta_k$ ; $\partial L / \partial w_{ij} = a_i\,\delta_j$, $\partial L / \partial b_j = \delta_j$ ; $w \leftarrow w - \eta\,\partial L / \partial w$. En matrices : $\boldsymbol{\Delta} = \mathbf{G} \odot f'(\mathbf{Z})$, $d\mathbf{W} = \mathbf{A}_{\text{prev}}^\top\boldsymbol{\Delta}$, $d\mathbf{b} = \mathbf{1}^\top\boldsymbol{\Delta}$, $\mathbf{G}_{\text{prev}} = \boldsymbol{\Delta}\mathbf{W}^\top$. Cross-entropy sur les logits : $(\mathbf{P} - \mathbf{Y})/n$ ; binaire : $(\sigma(\mathbf{z}) - \mathbf{y})/n$. Vérification : $\frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$, erreur relative $\frac{|a - n|}{\max(|a| + |n|, 10^{-12})}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 18.1 à 18.5 et 🧮 18.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un sens : `+1` (monter) ou `-1` (baisser) ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix ou une suite : les lettres collées, dans l'ordre demandé (`"CADB"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 18.6 à 18.8, la réflexion (🗣️ 18.10, 📄 18.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np
from scipy import special


def sigma(z):
    return special.expit(z)


# 18.3: L(w) = (a/2)(w - w*)^2 with a = 4, w* = 1, from w0 = 3
A_183, WSTAR_183, W0_183 = 4.0, 1.0, 3.0


def w_183(eta, t, curvature=A_183):
    return WSTAR_183 + (1 - eta * curvature) ** t * (W0_183 - WSTAR_183)


# 18.4 and 18.5: the tiny network X, Y -> A, B -> C, D (no bias), one example, label (1, 0)
X_184 = np.array([1.0, 2.0])
W1_184 = np.array([[0.5, 1.0], [-0.5, 0.25]])     # rows X, Y; columns A, B
W2_184 = np.array([[1.0, -1.0], [0.5, 0.5]])      # rows A, B; columns C, D
Y_184 = np.array([1.0, 0.0])


def tiny_184(W1, W2, f=lambda z: z, df=lambda z: np.ones_like(z)):
    """Forward and backward pass of the tiny network, loss 1/2 sum (out - y)^2, linear outputs."""
    z1 = X_184 @ W1
    a1 = f(z1)
    out = a1 @ W2
    loss = 0.5 * float(np.sum((out - Y_184) ** 2))
    d2 = out - Y_184
    d1 = (W2 @ d2) * df(z1)
    return dict(z1=z1, a1=a1, out=out, loss=loss, d2=d2, d1=d1, g1=np.outer(X_184, d1), g2=np.outer(a1, d2))


LIN_184 = tiny_184(W1_184, W2_184)
SIG_185 = tiny_184(W1_184, W2_184, f=sigma, df=lambda z: sigma(z) * (1 - sigma(z)))
GRAD2_184 = float(np.sum(LIN_184["g1"] ** 2) + np.sum(LIN_184["g2"] ** 2))

# 18.9: a 784-256-128-10 MLP on MNIST
SIZES_189 = [784, 256, 128, 10]
P_189 = sum(a * b + b for a, b in zip(SIZES_189[:-1], SIZES_189[1:]))
F_189 = sum(a * b for a, b in zip(SIZES_189[:-1], SIZES_189[1:]))      # multiplications of one forward pass

**18.Q1 — Pourquoi apprendre la rétropropagation si PyTorch la fait pour toi ?**

In [3]:
wb.record("18.Q1a", False, mistakes={"relis le §18.1 : qui calcule les gradients dans un projet réel ?": True})
wb.record("18.Q1b", "D", mistakes={"relis les quatre raisons du livre : la littérature et les échanges en font partie": "A",
          "relis les quatre raisons du livre : le diagnostic d'un réseau qui apprend mal en fait partie": "B",
          "relis les quatre raisons du livre : les idées qui s'appuient sur elle en font partie": "C"})
wb.record("18.Q1c", False, mistakes={"relis le début du §18.1 : un réseau n'est qu'un assemblage de petits calculs": True})

WB_ANSWER {"hash": "9b7f7c47efaa74b5b98ca4629b5e0996f562d0417676bd724e5e471443397f8c", "id": "18.Q1a", "kind": "bool", "mistakes": {"1f8101e1f4c9cd4ee7f0f4067f6ae19889d1a09dfbe50c9cc9ed44bdeac344e4": "relis le §18.1 : qui calcule les gradients dans un projet réel ?"}}
📝 Ex 18.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "f3f8d71b1c00a47424ec67f66cc84c3187e26853e2acafc21999654eb92d93bc", "id": "18.Q1b", "kind": "str", "mistakes": {"1af49ad69063de92d16743b72efe2d6e4d85040c46f17742fe6f4807fd503088": "relis les quatre raisons du livre : la littérature et les échanges en font partie", "8e467704b2c46aeaac027edb2312bdf86c5924c0ed43324117de89cc8f0b388b": "relis les quatre raisons du livre : les idées qui s'appuient sur elle en font partie", "f752469192d4550de2e75be04d525519661f22615f838289cf21b8cfa1a596f9": "relis les quatre raisons du livre : le diagnostic d'un réseau qui apprend mal en fait partie"}}
📝 Ex 18.Q1b : réponse enregistrée (str).
WB_ANSWER {"hash": "c502dd7a9f4710c4076abeb

**18.Q2 — Perturber un poids au hasard : pourquoi est-ce si lent ?**

In [4]:
wb.record("18.Q2a", 1_000_000, mistakes={"un essai par poids, une passe avant par essai : la passe de départ ne compte pas ici": 1_000_001,
          "un seul essai par poids, et une seule passe avant par essai": 2_000_000,
          "combien de poids faut-il essayer, chacun une fois ?": 1})
wb.record("18.Q2b", 0.5, decimals=2, mistakes={"deux sens possibles, tirés au hasard : lequel fait baisser l'erreur ?": 0.25,
          "une proportion entre 0 et 1 : combien de sens possibles, et combien de bons ?": 1.0})
wb.record("18.Q2c", True, mistakes={"relis la p. 714 : un poids sert à tous les exemples": False})
wb.record("18.Q2d", "B", mistakes={"relis la fin du §18.2.1 : l'astuce du biais du ch. 10": "A",
          "le biais aussi doit être appris": "C", "c'est l'entrée du biais qui vaut 1, pas le biais lui-même": "D"})

WB_ANSWER {"hash": "0195adc408e3595b7bc3424f331f39398ebe0c2310af0072cfc6acbc86d0a037", "id": "18.Q2a", "kind": "int", "magnitude": 6, "mistakes": {"5b75f37ea49a9550c6fa1f7dc29d9d0af577bb6e9d0dce80e0ce689e8e13d4b4": "combien de poids faut-il essayer, chacun une fois ?", "bcc66320858eabd63883b021faa9a6846b103c74513a9546b61529bebf6c5116": "un essai par poids, une passe avant par essai : la passe de départ ne compte pas ici", "edbd478231a02989c0eaaab221f54f0d352ee3d8e578c4a960a02ba2a4cb5f60": "un seul essai par poids, et une seule passe avant par essai"}, "sign": 1}
📝 Ex 18.Q2a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "ad1273df52e78f96fbf254c4464fe40a583155c4392a857ad7f92c24704d5daa", "hash_coarse": "41e83d51cab5ea3c0ee47e3570c2ca65cf9a7b611cae17d10416e373c8a2db8a", "id": "18.Q2b", "kind": "float", "magnitude": -1, "mistakes": {"ab150b30f65c84999a04526af9d0b2bb1fa6142af63b98fa1d6ec7e662387859": "deux sens possibles, tirés au hasard : lequel fait baisser l'erreur ?", "

**18.Q3 — Ce que le gradient dit à chaque poids, et le cas du biais**

In [5]:
wb.record("18.Q3a", 1, mistakes={"une pente négative : que fait l'erreur quand le poids augmente ?": -1,
          "il faut choisir un sens : +1 ou −1": 0})
wb.record("18.Q3b", -1, mistakes={"une pente positive : que fait l'erreur quand le poids augmente ?": 1,
          "il faut choisir un sens : +1 ou −1": 0})
wb.record("18.Q3c", True, mistakes={"relis la fin du §18.2.2 : c'est justement le pari du livre, à petits pas": False})
wb.record("18.Q3d", "B", mistakes={"les activations usuelles ont une dérivée (presque) partout": "A",
          "le livre remet les activations au §18.10": "C", "relis le ch. 17 : un réseau sans activation s'effondre": "D"})

WB_ANSWER {"hash": "a8038e804bc47c7b29f220bd9197d75a4d2696ae878fd883373bfd90c3e83fab", "id": "18.Q3a", "kind": "int", "magnitude": 0, "mistakes": {"4f350b7c62b2d6e441e3fbc50fb377bc02e12c04a4fd72210ce50d422026033b": "il faut choisir un sens : +1 ou −1", "cb1b9cfc19d48db012729038cfd5151e9f40c6d696bfee4ec49dbb57cf65ea79": "une pente négative : que fait l'erreur quand le poids augmente ?"}, "sign": 1}
📝 Ex 18.Q3a : réponse enregistrée (int).
WB_ANSWER {"hash": "fb720b2f1364c25fbefa23784de276a74365d3f03952338ba3a741ce80b404ed", "id": "18.Q3b", "kind": "int", "magnitude": 0, "mistakes": {"be8c96ed32859ff7f4297859f7e688bfe3345557f962c59db8af7b0799aa91e6": "une pente positive : que fait l'erreur quand le poids augmente ?", "ddfb70aa35f5120b5134f3b32be727a10ed8743e123ff0c0cdf3a3079743b85d": "il faut choisir un sens : +1 ou −1"}, "sign": -1}
📝 Ex 18.Q3b : réponse enregistrée (int).
WB_ANSWER {"hash": "188d958c6179f96d2e02111e44c08f17da5550163396499efa7b37426c9aea4f", "id": "18.Q3c", "kind": "boo

**18.Q4 — Le delta d'un neurone : signe et effet sur l'erreur**

In [6]:
wb.record("18.Q4a", "A", mistakes={"le produit de deux nombres de même signe est positif : l'erreur change dans quel sens ?": "B",
          "le delta est le facteur entre les deux changements : l'erreur bouge": "C"})
wb.record("18.Q4b", 1, mistakes={"il faut un changement de signe opposé à celui du delta": -1,
          "il faut choisir un sens : +1 ou −1": 0})
wb.record("18.Q4c", False, mistakes={"relis le §18.4.1 : de 7 à 7,5 ou de 700 à 700,5, l'effet est-il différent ?": True})
wb.record("18.Q4d", 0.4 * -2.5, decimals=2, mistakes={"le delta multiplie le changement de la sortie, pas sa nouvelle valeur": -2501.0,
          "le changement de la sortie est 1 000,4 − 1 000": -0.25})
wb.record("18.Q4e", True, mistakes={"relis le §18.4 : deux étapes, dont une seule est la rétropropagation au sens strict": False})

WB_ANSWER {"hash": "f145f8f6db9fd6f6294899d0beb19205a34bb529f8fd0b641c1b7d0f8470e56d", "id": "18.Q4a", "kind": "str", "mistakes": {"90da0430bc0677185576da988a25b6297a79df7d6b383fae73f61b98e6507cdf": "le delta est le facteur entre les deux changements : l'erreur bouge", "9c03e3d0595f8a3b7914a77a9e32aa4576990fb06bfe8bca04f9a1efee36f8d8": "le produit de deux nombres de même signe est positif : l'erreur change dans quel sens ?"}}
📝 Ex 18.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "60b8f00a221c0ef2fd9fddd4778c41e28c8fceb2dd3a800a3ca2de3d7d2da44a", "id": "18.Q4b", "kind": "int", "magnitude": 0, "mistakes": {"6a7674016a6b5ad02bae683125f3735c3486d12dbd995df54eee6a97ff70b841": "il faut choisir un sens : +1 ou −1", "901524d0ebf8f509b575fb8d13d3b80cb2064663394ecc6c0f70a9f3e2a2fc2d": "il faut un changement de signe opposé à celui du delta"}, "sign": 1}
📝 Ex 18.Q4b : réponse enregistrée (int).
WB_ANSWER {"hash": "8a96bd3cefa5129257dd58dda453697ba18fe51da5d41367376075e593cfbbe2", "id": "18.

**18.Q5 — Le réseau minuscule : couches, poids et leurs noms**

In [7]:
wb.record("18.Q5a", 8, mistakes={"le réseau minuscule n'a pas de biais": 12, "compte les fils : chaque entrée va à A et à B, chaque caché à C et à D": 4,
          "compte les fils entre les couches, pas les neurones": 6})
wb.record("18.Q5b", "A", mistakes={"le neurone de départ vient en premier dans le nom": "B", "c'est la notation d'un delta": "C",
          "c'est la notation d'une sortie": "D"})
wb.record("18.Q5c", 2, mistakes={"la couche d'entrée ne calcule rien et ne compte pas": 3, "une couche cachée et une couche de sortie": 1})
wb.record("18.Q5d", "B", mistakes={"le réseau prédit la plus grande des deux sorties": "A",
          "ce ne sont pas des probabilités, mais la plus grande sortie désigne quand même une classe": "C"})
wb.record("18.Q5e", [0, 0, 1, 0], mistakes={"les indices commencent à 0 : l'indice 2 est la troisième case": [0, 1, 0, 0],
          "quatre classes d'indices 0 à 3 : où est l'indice 2 ?": [0, 0, 0, 1]})

WB_ANSWER {"hash": "58601e530e78d5a2d11bef83b7d2f15e88322ee134ced8d46cb89dfa6e518f21", "id": "18.Q5a", "kind": "int", "magnitude": 0, "mistakes": {"1e64ba9f29e4f77dab24eb86c807ae9c9242c946ed6ac72fbe0a8eeb16e56137": "compte les fils entre les couches, pas les neurones", "e7b47c22371eab6e198484933f555b7ea55dc3715a1de52651b465ea726064eb": "le réseau minuscule n'a pas de biais", "e93b9bb6d8b286c710bec700f5f96cddb9c0020a3fed54e0125667c566af1785": "compte les fils : chaque entrée va à A et à B, chaque caché à C et à D"}, "sign": 1}
📝 Ex 18.Q5a : réponse enregistrée (int).
WB_ANSWER {"hash": "b203fa11e816d84e408ed189583d7f2d40bf9957bfdc8baeee6f7c05df410be7", "id": "18.Q5b", "kind": "str", "mistakes": {"6e009b78d2c980528b68c6a2932467b166401ed9ddedfe6eb53feb1c01074836": "le neurone de départ vient en premier dans le nom", "91d1fc6ca66335f1d80072e74cc35b84146ab1d065d655a491b27dfe0cb3d65a": "c'est la notation d'une sortie", "9e5eb368d9fa642ff4f96f424208dc48ab8962fa8f72469b8d33d6c0f0cb2a23": "c'es

**18.Q6 — Delta de sortie, delta caché, mise à jour d'un poids**

In [8]:
wb.record("18.Q6a", "A", mistakes={"un petit changement du poids est multiplié par a avant d'arriver au neurone": "B",
          "relis le §18.7 : la sortie de départ multiplie, elle ne divise pas": "C",
          "c'est le facteur d'un chemin dans le delta d'un neurone caché": "D"})
wb.record("18.Q6b", 1 - 0.1 * 2 * -0.5, decimals=2, mistakes={"δ est négatif : retirer η a δ fait monter le poids": 0.9,
          "as-tu multiplié par la sortie a ?": 1.05, "as-tu multiplié par η ?": 2.0})
wb.record("18.Q6c", 0.5 * 2 + -1 * 3, decimals=2, mistakes={"le poids −1 garde son signe": 4.0, "il manque un des deux chemins": 1.0,
          "chaque delta est multiplié par le poids de son fil, puis on additionne": 2.5})
wb.record("18.Q6d", True, mistakes={"relis le §18.8 : les couches plus lointaines interviennent-elles directement ?": False})

WB_ANSWER {"hash": "416eac938f68759a37eb254df549cdec64ccd3ee343b99842bf56010a909f797", "id": "18.Q6a", "kind": "str", "mistakes": {"13e3074590d1f5df98f608696e23ff7e76ea7bfd522d5e270d83c818cb49d8ab": "relis le §18.7 : la sortie de départ multiplie, elle ne divise pas", "d7681ba707977ac8d46a6b5e97d932004101a8e674fd791704e1edf5b7d151b5": "c'est le facteur d'un chemin dans le delta d'un neurone caché", "fdeeefc44e455d6abae6a09622141425fffa065d8f3fbe98d03b64586edc7aea": "un petit changement du poids est multiplié par a avant d'arriver au neurone"}}
📝 Ex 18.Q6a : réponse enregistrée (str).
WB_ANSWER {"decimals": 2, "hash": "d4d93f4ab6db3ab249761c8bdcce5bb3aee7e89aa47a3b2944eeb522d3117d30", "hash_coarse": "2e4cba4c8d98db1ecbd0dba62256e86f5f5e83d2352179b137921639053096f0", "id": "18.Q6b", "kind": "float", "magnitude": 0, "mistakes": {"3fa89e79341ea34400ee91abb3687f1e74d83198f54f38a6f586b91daddea460": "δ est négatif : retirer η a δ fait monter le poids", "5e8af365a77560224eeb102b3e925ea79cc70d7

**18.Q7 — Rétropropager couche par couche, puis avec une activation**

In [9]:
wb.record("18.Q7a", "D", mistakes={"les deltas partent de la fin du réseau": "A", "relis le §18.9 : où l'erreur est-elle connue directement ?": "B",
          "relis le §18.9 : quelle couche n'a pas de couche suivante ?": "C"})
wb.record("18.Q7b", 3, mistakes={"on recule d'une seule couche depuis la sortie": 4, "on recule d'une couche, pas jusqu'au début": 2})
wb.record("18.Q7c", "B", mistakes={"c'est la valeur de l'activation, pas sa pente": "A", "les dérivées du ch. 17 se prennent en z": "C",
          "une pente forte amplifie : on multiplie par la pente": "D"})
wb.record("18.Q7d", 0.25, decimals=2, mistakes={"c'est σ(0) ; le facteur est la dérivée": 0.5, "σ′(z) = σ(z)(1 − σ(z))": 1.0})
wb.record("18.Q7e", True, mistakes={"la dérivée de l'identité vaut 1 : que change la multiplication ?": False})

WB_ANSWER {"hash": "b5fb33aac26c432b4542b33d10955eebcc87140f5081d4c2cca0d61f6e34b954", "id": "18.Q7a", "kind": "str", "mistakes": {"117617877d0313fc7cce2a09d8551527d1bc93809e91f8fbe694d9cf269f26a5": "les deltas partent de la fin du réseau", "1c4a4f6cd83ab47c676cf7b9aa9b06a3445d00bc3f72f5c85c68decb50ebffc3": "relis le §18.9 : quelle couche n'a pas de couche suivante ?", "2bb2ed8302c54e3a2a36cfeb19bd69f22307f2827908ab61c715d6d4084b7ff7": "relis le §18.9 : où l'erreur est-elle connue directement ?"}}
📝 Ex 18.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "3acdfe00ff506103c2e69fcb5d2d3d718244b20d65c80921e1c42b9cd0cee5f6", "id": "18.Q7b", "kind": "int", "magnitude": 0, "mistakes": {"6c6a1d0a58b12e18f53dcee2c24a4af84cc71fef67379a8c35aab813ef662c56": "on recule d'une couche, pas jusqu'au début", "a53fdf2494e8d0c9052faaedb5c684422b546b3a77b5c68f03e77dcdff2aebdd": "on recule d'une seule couche depuis la sortie"}, "sign": 1}
📝 Ex 18.Q7b : réponse enregistrée (int).
WB_ANSWER {"hash": "db123

**18.Q8 — Learning rate trop grand ou trop petit : reconnaître les symptômes**

In [10]:
wb.record("18.Q8a", "A", mistakes={"des poids qui sautent d'un bord à l'autre : des pas trop courts ou trop longs ?": "B",
          "une loss qui ne baisse jamais n'est pas un bon réglage": "C",
          "relis le §18.11.1 : le symptôme vient du pas, les données sont les mêmes": "D"})
wb.record("18.Q8b", "B", mistakes={"une loss qui baisse régulièrement ne diverge pas": "A", "bien choisi, il irait plus vite": "C",
          "la loss baisse : les données ne sont pas en cause": "D"})
wb.record("18.Q8c", True, mistakes={"deux classes équilibrées : quelle accuracy pour un réseau qui répond toujours pareil ?": False})
wb.record("18.Q8d", "B", mistakes={"relis la p. 785 : le livre, d'après Dauphin et coll., penche pour une autre explication": "A",
          "le réseau finit par repartir : rien n'est cassé": "C", "c'est un comportement de la surface de la loss": "D"})
wb.record("18.Q8e", True, mistakes={"η multiplie chaque correction : que vaut-elle pour η = 0 ?": False})

WB_ANSWER {"hash": "d0bc49b0366a7301b5b4965f868959bb7149b3c5c3f32ddf6fbc705848829199", "id": "18.Q8a", "kind": "str", "mistakes": {"65d398e8d7172618cec132b2d2ad5eb0f51fea275f6dad027ba9f0626c393496": "une loss qui ne baisse jamais n'est pas un bon réglage", "8f29c9fd9323a180ca0069ab87c5271a8c0dbd3ee1d2e0cc482ee5e49dcc57f4": "relis le §18.11.1 : le symptôme vient du pas, les données sont les mêmes", "924e34bb00f40020ce4e612b374c6510869edbc48802cf33d0bda0220e6c119b": "des poids qui sautent d'un bord à l'autre : des pas trop courts ou trop longs ?"}}
📝 Ex 18.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "58f4f5b1091906ece25e61f1171f540857063b9ebf2942d30c3f02cab750899e", "id": "18.Q8b", "kind": "str", "mistakes": {"0ab617e862f0efd9e713bcb98c283a7a5c43d6a3d07607652a7d63c8c617b02a": "la loss baisse : les données ne sont pas en cause", "1ed32df745335f356585e31b1d2d5604295bac1e9b9d92997d370f6ad9311cde": "une loss qui baisse régulièrement ne diverge pas", "399b859636bf0e68da3c4cd772b6e2011

**18.Q9 — Ce que la rétropropagation propage vraiment**

In [11]:
wb.record("18.Q9a", False, mistakes={"relis le §18.12.2 : l'erreur sert-elle ailleurs qu'au départ ?": True})
wb.record("18.Q9b", "B", mistakes={"relis le §18.12.3 : un delta est un taux de variation": "A", "un delta est porté par un neurone, pas par un fil": "C",
          "la sortie et le delta sont deux nombres différents": "D"})
wb.record("18.Q9c", "BDCA", mistakes={"les deltas se calculent de la sortie vers l'entrée": "BCDA", "il faut d'abord une passe avant": "DBCA",
          "les poids ne changent qu'une fois tous les deltas connus": "BDAC"})
wb.record("18.Q9d", False, mistakes={"qui fixe la taille du pas : la rétropropagation ou le learning rate ?": True})

WB_ANSWER {"hash": "4e06108ef65b03178d2b36bcc7c62732c672601379e5014d4ad73e6522e1100b", "id": "18.Q9a", "kind": "bool", "mistakes": {"dda8ecca745506612d9ad92944b8ec365a3148957059ab91e5d5f029aa322477": "relis le §18.12.2 : l'erreur sert-elle ailleurs qu'au départ ?"}}
📝 Ex 18.Q9a : réponse enregistrée (bool).
WB_ANSWER {"hash": "27379a13387b4dc13c3e307c2c8818b490b631225f22092d2f1e4427df824c55", "id": "18.Q9b", "kind": "str", "mistakes": {"283853303e772ce65e9fa728ca0c766bb1002c9e4187a0eab6d27c3e2e50cbee": "un delta est porté par un neurone, pas par un fil", "2bec98a82b42e578694a03d097053497bba879b3732f4f9624d73dbe9995efeb": "relis le §18.12.3 : un delta est un taux de variation", "f5ab690cd74a99d24fc15eba2d856cc5c0b13811674fe01dd7bf4b505da33406": "la sortie et le delta sont deux nombres différents"}}
📝 Ex 18.Q9b : réponse enregistrée (str).
WB_ANSWER {"hash": "1298c6e00d4c49e1d81b75b9748a96f1bf314489c705930cb735d76469b7270e", "id": "18.Q9c", "kind": "str", "mistakes": {"08248c966030f33b1e

**18.Q10 — Neurones saturés et ReLU mortes**

In [12]:
wb.record("18.Q10a", True, mistakes={"σ′(15) = σ(15)(1 − σ(15)) et 1 − σ(15) ≈ e^(−15)": False})
wb.record("18.Q10b", 0.0, decimals=2, mistakes={"la pente 1 de ReLU ne vaut que pour z > 0": 1.0, "la pente de ReLU n'est jamais négative": -1.0,
          "c'est σ′(0), la dérivée de la sigmoïde ; on demande celle de ReLU": 0.25, "c'est σ(0) ; on demande la dérivée de ReLU": 0.5})
wb.record("18.Q10c", "ACD", mistakes={"il en manque une : relis les remèdes du §18.12.4": "AC", "il en manque une : relis la fin du §18.12.4": "CD",
          "il en manque une : relis les remèdes proposés par le livre et par la fiche": "AD",
          "un grand learning rate pousse les poids loin de 0": "ABCD", "plus de sigmoïdes, c'est plus de neurones qui peuvent saturer": "ACDE"})
wb.record("18.Q10d", True, mistakes={"pour z > 0, la pente de ReLU vaut 1 : y a-t-il un plateau ?": False})

WB_ANSWER {"hash": "6d445ed93844549efac25012c6fb94d222e625b987fc2dae881da0b7f21a5e1f", "id": "18.Q10a", "kind": "bool", "mistakes": {"9cb892e4aceb7345e785c1084d2d6459b44d4dfe7a4fca28254e9a3ee43a7d56": "σ′(15) = σ(15)(1 − σ(15)) et 1 − σ(15) ≈ e^(−15)"}}
📝 Ex 18.Q10a : réponse enregistrée (bool).
WB_ANSWER {"decimals": 2, "hash": "0208a0a14a74e5f17e002c394e0f0609d36a77b2795310880bcb3e11dd0fa884", "hash_coarse": "d4a47b6462d6b3c0c39df51df9fa11becf5eed24c5ffc05c1364f91a0546a9fa", "id": "18.Q10b", "kind": "float", "magnitude": null, "mistakes": {"23445dca059bfc9cff6fb0fabc9311e275f6eda095171a0b64a7d3a76607355f": "c'est σ′(0), la dérivée de la sigmoïde ; on demande celle de ReLU", "263f0820095ad63e8f691cf4ade1c7c967c7928f0c06f763dca3480e0815fe1a": "c'est σ(0) ; on demande la dérivée de ReLU", "50ca0fe751873592ed54a604cc825431b2191ffd8d724ad9fab2555403f37311": "la pente de ReLU n'est jamais négative", "fc73b28f81486472e84f87bf0680d280828fe19d608b65f0a90dcb2d968633d3": "la pente 1 de ReLU ne 

**18.Q11 — Mini-batches, mises à jour parallèles et coût du backward**

In [13]:
wb.record("18.Q11a", 6400 // 32, mistakes={"une mise à jour par mini-batch, pas par exemple": 6400, "c'est la taille du mini-batch": 32,
          "6 400 / 32": 199})
wb.record("18.Q11b", False, mistakes={"relis le §18.12.5 : quand se fait la mise à jour ?": True})
wb.record("18.Q11c", "B", mistakes={"les deltas coûtent une passe avant ; et les gradients des poids ?": "A", "c'est l'ordre de grandeur des différences finies, pas de la rétropropagation": "C",
          "le coût reste proportionnel au nombre de poids": "D"})
wb.record("18.Q11d", True, mistakes={"relis le §18.12.6 : de quoi dépend la mise à jour d'un poids ?": False})

WB_ANSWER {"hash": "559b893edf979c8c5ff8147944efb30f7537ee449165353067e9ff666fedcfed", "id": "18.Q11a", "kind": "int", "magnitude": 2, "mistakes": {"1b3cab684b4f2dfb0d38329d4d11bf83f203206268fbcb79d00819367e095fed": "c'est la taille du mini-batch", "4ea131426a611034aeca909dd086c77b7b8f8ba74a3b69c9ba903245c1539c48": "6 400 / 32", "f15b774cfdf052512a9feb46bc6146dd5643d5750bcb9c93c5869e9e1a7d71cb": "une mise à jour par mini-batch, pas par exemple"}, "sign": 1}
📝 Ex 18.Q11a : réponse enregistrée (int).
WB_ANSWER {"hash": "3249f6dd7b0e4e4e04067ccf2feb018ae1df52975cb04beb177f1b7123ac2705", "id": "18.Q11b", "kind": "bool", "mistakes": {"10a368179e94cbef1b6668382f5436cef51075583253191a718898ec5588027e": "relis le §18.12.5 : quand se fait la mise à jour ?"}}
📝 Ex 18.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "66342e1c14e38a2293060aebf419655af84eea14034c08b231b66bdf5d29d56a", "id": "18.Q11c", "kind": "str", "mistakes": {"a7818e273f31b76070f14b529fec48c3540d6254b897e841f1855b0ac7c7813d

**18.Q12 — Aucune garantie, une longue histoire, des équations à apprivoiser**

In [14]:
wb.record("18.Q12a", False, mistakes={"relis le §18.12.8 : le perceptron a une preuve, la rétropropagation non": True})
wb.record("18.Q12b", "B", mistakes={"relis le §18.12.8 : on commence par calmer les choses": "A", "relis le §18.12.8 : le réseau vient en second": "C",
          "relis le §18.12.8 : le premier réflexe porte sur un réglage": "D"})
wb.record("18.Q12c", 1970, mistakes={"c'est l'année de sa publication en anglais ; on demande le mémoire": 1976, "c'est la thèse de Werbos": 1974,
          "c'est l'article de Rumelhart, Hinton et Williams": 1986, "c'est la date des travaux d'Ostrowski": 1971})
wb.record("18.Q12d", 1986, mistakes={"c'est la publication de Werbos sur les réseaux": 1982, "c'est le mémoire de Linnainmaa": 1970})
wb.record("18.Q12e", "B", mistakes={"vérifie les formes : (n, n_out) fois (n_in, n_out) ne se multiplie pas": "A",
          "aucune inverse : on transporte des gradients": "C", "relis le §18.12.10 : un seul produit par une matrice de poids": "D"})

WB_ANSWER {"hash": "d5de9569fc7730e88b5dc08a401dd40e69bba48b3096dbaeb2a98d2c99fe29d7", "id": "18.Q12a", "kind": "bool", "mistakes": {"4f5db0bb510e2e64102a63962151d553bdc0e92f374432dd6d45fb87ec761c0b": "relis le §18.12.8 : le perceptron a une preuve, la rétropropagation non"}}
📝 Ex 18.Q12a : réponse enregistrée (bool).
WB_ANSWER {"hash": "6ce745624cf81e95e27a3d3627a299dadb2d8d1b9d3117fcd7ba072caf5eb3b5", "id": "18.Q12b", "kind": "str", "mistakes": {"3b5b3f4118e6f4235b0efc0f1a7e449df6778a10ff7de06fa603b97b69665f20": "relis le §18.12.8 : le premier réflexe porte sur un réglage", "b232edabb44c53d5f11d95bca7d1bcf01d371a4cc4e4c0948ed28fbeada9f016": "relis le §18.12.8 : le réseau vient en second", "facc34fea1535ff4e8995c1b0135de1f2cb7c3bf70c43177dcd405db4b6af7db": "relis le §18.12.8 : on commence par calmer les choses"}}
📝 Ex 18.Q12b : réponse enregistrée (str).
WB_ANSWER {"hash": "d30db4b345d3b6132ebe51a21f1a4820b9680946acbea7f6cb9242e2b91bf664", "id": "18.Q12c", "kind": "int", "magnitude": 

**18.R1 — La dérivée de la sigmoïde à partir de sa sortie**

In [15]:
wb.record("18.R1a", 0.8 * 0.2, decimals=2, mistakes={"σ′ = σ(1 − σ), avec σ = 0,8": 0.2, "c'est la formule de tanh′": 0.36})
wb.record("18.R1b", 0.5 * 0.5, decimals=2, mistakes={"σ′ = σ(1 − σ), avec σ = 0,5": 0.5})
wb.record("18.R1c", 1 - 0.6 ** 2, decimals=2, mistakes={"tanh′ = 1 − tanh², pas 1 − tanh": 0.4, "c'est la formule de la sigmoïde": 0.24, "c'est tanh²(z), pas encore la dérivée : revois la formule de tanh′ au ch. 17": 0.36})
wb.record("18.R1d", True, mistakes={"σ′ s'écrit avec σ seul : a(1 − a)": False})
wb.record("18.R1e", 0.0, decimals=2, mistakes={"la pente 1 de ReLU ne vaut que pour z > 0": 1.0, "c'est z ; on demande la pente de ReLU en z": -0.3,
          "c'est σ′(−0,3), la dérivée de la sigmoïde ; on demande celle de ReLU": float(sigma(-0.3) * (1 - sigma(-0.3))),
          "c'est σ(−0,3) ; on demande la dérivée de ReLU": float(sigma(-0.3))})

WB_ANSWER {"decimals": 2, "hash": "ee05b26c3dc3ba3c573ac101c6bdfa0b016ae6281f5a3038a78002606d962dc6", "hash_coarse": "3fc1cdf3e4e8d9c15ea57785a130df0721eabe30c9ff9ab3fda073c05dbe3323", "id": "18.R1a", "kind": "float", "magnitude": -1, "mistakes": {"2f7005e57aae7bce4800e2c3a41a9bf7e5257b07e92f4379febb334fb7023f7a": "σ′ = σ(1 − σ), avec σ = 0,8", "3c90b42e5e617088fb2be1dc40149e22c4d1fa133fa79b21ddf6e9fd0b4887ce": "c'est la formule de tanh′"}, "sign": 1}
📝 Ex 18.R1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"alt_coarse": ["b7cb75487d91b3204de766df096f13d4f7f5fcf7c0ee221ec21cc386cf3d8726"], "decimals": 2, "hash": "59fa889cfebaf93068f17a0ecd6eabf5c5eac360a8c519560e4e8dd26b1dc89f", "hash_coarse": "18b10e97766c1886dfe9798a6b524b2e110961d383586bacc8af738925c6d381", "id": "18.R1b", "kind": "float", "magnitude": -1, "mistakes": {"02ff8a58526d9e699a54ed27e0078e930f554fa7a223550887b0bd5ad270c0a0": "σ′ = σ(1 − σ), avec σ = 0,5"}, "sign": 1}
📝 Ex 18.R1b : réponse enregistrée (float, 2

**18.R2 — Chercher le meilleur learning rate par recherche en grille**

In [16]:
wb.record("18.R2a", 4 * 5 + 1, mistakes={"refit=True réentraîne une fois de plus sur tout l'entraînement": 20, "chaque valeur est testée sur chacun des 5 folds": 5,
          "chaque valeur est testée sur 5 folds, puis le meilleur réglage est réentraîné": 4})
wb.record("18.R2b", "A", mistakes={"scikit-learn accepte n'importe quelle liste de valeurs": "B", "il y a autant de valeurs que dans une grille linéaire de même taille": "C",
          "GridSearchCV fait la validation croisée dans tous les cas": "D"})
wb.record("18.R2c", "A", mistakes={"ce sont les poids appris, pas le réglage": "B", "c'est le premier réglage essayé": "C", "c'est une méthode qui note un modèle": "D"})
wb.record("18.R2d", True, mistakes={"relis la documentation de MLPClassifier : le solveur lbfgs a-t-il un learning rate ?": False})

WB_ANSWER {"hash": "0682fbf46918509fd59da8da885d88c22b0802cda0daea5d59d3550889b4795b", "id": "18.R2a", "kind": "int", "magnitude": 1, "mistakes": {"a39ae5ced9f95a75fd8d678554b2f8d0b27af77ccede5869679abad5fabc4f0f": "chaque valeur est testée sur 5 folds, puis le meilleur réglage est réentraîné", "b3f47a39d09d566118ccda81a2194bd7201f64fc75329a18ed19a6c9d5a86770": "chaque valeur est testée sur chacun des 5 folds", "be61c31fbf1ec5e9a1beac294990a8f15d85f3be2da1bbf283ea13c151089af9": "refit=True réentraîne une fois de plus sur tout l'entraînement"}, "sign": 1}
📝 Ex 18.R2a : réponse enregistrée (int).
WB_ANSWER {"hash": "882d1a9366957b49111b3cbfce9270cac0454dc03c3f08310714eb108490a461", "id": "18.R2b", "kind": "str", "mistakes": {"13cae763ec2de9c7fe2de157391a0a276a5e26d31e327bc66fb658f77dbf5272": "scikit-learn accepte n'importe quelle liste de valeurs", "4d89e75d3297dac57e96fe72389c2c458f48e189e3cc5fd060fcee29f348c43a": "il y a autant de valeurs que dans une grille linéaire de même taille", "

**18.R3 — Représentation, évaluation, optimisation dans un MLP**

In [17]:
wb.record("18.R3a", "B", mistakes={"la loss ne dit pas ce que le modèle peut exprimer": "A", "la loss est ce qu'on minimise, pas la méthode": "C"})
wb.record("18.R3b", "C", mistakes={"la descente ne change pas ce que le modèle peut exprimer": "A", "elle ne juge pas une solution, elle en cherche une": "B"})
wb.record("18.R3c", "A", mistakes={"l'architecture ne juge pas une solution": "B", "l'architecture ne cherche rien : elle fixe ce qui est possible": "C"})
wb.record("18.R3d", True, mistakes={"ce qu'une représentation ne peut pas exprimer peut-il s'apprendre ? (ch. 11)": False})

WB_ANSWER {"hash": "aca4d921d75ce082019e9efc7d8e70a0fa95f29c6c72a6a2b9e9bc9121d6894f", "id": "18.R3a", "kind": "str", "mistakes": {"2dd9c86ca35e7f472a60e7860f16dd1c53399281ec6f5c15eef19f124869c246": "la loss ne dit pas ce que le modèle peut exprimer", "a66585a2c3ec56689992a2a54c721d411846d01a8143ee7cf228ef702cab27b0": "la loss est ce qu'on minimise, pas la méthode"}}
📝 Ex 18.R3a : réponse enregistrée (str).
WB_ANSWER {"hash": "c9fde66a4062922049058f59d3292d6f4bb4972cd83e57e6b1c2aa13e1b5e4e4", "id": "18.R3b", "kind": "str", "mistakes": {"08dcca68c819b30f9b4be0abf32511e198c9ca77508861163a48c958903f5914": "elle ne juge pas une solution, elle en cherche une", "43de19e08e95a0fed00416f2f083f14673d4a5e2e213ce7525627dd8b9c35db6": "la descente ne change pas ce que le modèle peut exprimer"}}
📝 Ex 18.R3b : réponse enregistrée (str).
WB_ANSWER {"hash": "186b442ce85adb0f31f31c1203a8312e0d93c062caade87c13fe3c07b38b1bba", "id": "18.R3c", "kind": "str", "mistakes": {"2452ed617b63289fe472ceffb002a3ad51

**Ex 18.1 — Le delta, amplificateur de changement**

In [18]:
wb.record("18.1a", -3 * 0.3, decimals=2, mistakes={"le delta multiplie le changement de la sortie, pas sa nouvelle valeur": -4.5,
          "le changement de la sortie est 1,5 − 1,2": -0.3})
wb.record("18.1b", 0.5 * -0.4, decimals=2, mistakes={"le delta multiplie le changement de la sortie, pas sa nouvelle valeur": 1.8,
          "le changement de la sortie est 3,6 − 4": -0.4})
wb.record("18.1c", -0.06 / -1.5, decimals=2, mistakes={"un delta négatif : pour faire baisser l'erreur, la sortie doit-elle monter ou baisser ?": -0.04,
          "ΔE = δ × Δo : pour trouver Δo, divise": 0.09})
wb.record("18.1d", 2.0 * (3 - 2), decimals=1, mistakes={"la dérivée de (o − 2)² est 2(o − 2)": 1.0, "c'est la dérivée de o − 2 seul ; on dérive un carré": 0.5})
wb.record("18.1e", 2 * 0.1, decimals=2, mistakes={"multiplie le delta par le changement de la sortie, 0,1": 2.0, "relis la question : on demande la prévision par le delta": 0.21})
wb.record("18.1f", (3.1 - 2) ** 2 - (3 - 2) ** 2, decimals=2, mistakes={"relis la question : on demande le vrai changement, E(3,1) − E(3)": 0.2,
          "c'est E(3,1) ; on demande le changement depuis E(3)": 1.21})
wb.record("18.1g", False, mistakes={"le delta est une pente : que dit son signe ?": True})

WB_ANSWER {"decimals": 2, "hash": "4337d4251f958e00563fa3abb8e95d8a639ef72f7a6880d099552a9c29572ae6", "hash_coarse": "42f28d0851f67c05ea08978040f058beb3d48139494ca35f061fe51bd5a68530", "id": "18.1a", "kind": "float", "magnitude": -1, "mistakes": {"3bb5feb432099a2da8d8dc817db8163a226ef06ed9625443590531785139fa8a": "le delta multiplie le changement de la sortie, pas sa nouvelle valeur", "74f82927afab5cecb7abc957b13787f2af97198efc39f22b6534a0ad0754230a": "le changement de la sortie est 1,5 − 1,2"}, "sign": -1}
📝 Ex 18.1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "2b0d892d4661c256d5dbf8e6a4f92f896ecef5f62645dc194979cb19427b9ae5", "hash_coarse": "9e28cd2b34b98b39cae49de790b541ea93d5aade98c2142b4b0d56a9536ee5ad", "id": "18.1b", "kind": "float", "magnitude": -1, "mistakes": {"2066fb15149adc50131b925d4772527ef7418589239903a9ba04a9d326471ba6": "le delta multiplie le changement de la sortie, pas sa nouvelle valeur", "b967a992f898b22a2252306c3b04587434070010b

**Ex 18.2 — Deltas de sortie avec l'erreur quadratique : attention au signe**

In [19]:
wb.record("18.2a", 0.5 * (0.6 ** 2 + 1.2 ** 2), decimals=2, mistakes={"n'oublie pas le facteur ½": 1.8, "relis le label : y = (0 ; 1)": 0.2})
wb.record("18.2b", 0.6 - 0, decimals=2, mistakes={"δ = ŷ − y, pas y − ŷ": -0.6, "avec le ½, la dérivée de ½(ŷ − y)² est ŷ − y": 1.2})
wb.record("18.2c", -0.2 - 1, decimals=2, mistakes={"δ = ŷ − y, pas y − ŷ": 1.2, "avec le ½, la dérivée de ½(ŷ − y)² est ŷ − y": -2.4})
wb.record("18.2d", -1.2 * 0.01, decimals=3, mistakes={"un delta négatif et une sortie qui monte : l'erreur monte-t-elle ?": 0.012,
          "multiplie le delta par le changement, 0,01": -1.2})
wb.record("18.2e", 0.5 * ((0.6 - 0.5 * 0.6) ** 2 + (-0.2 + 0.5 * 1.2 - 1) ** 2), decimals=3, mistakes={"n'oublie pas le facteur ½": 0.45, "ŷ ← ŷ − 0,5 δ : le second delta est négatif, ŷ₂ monte": 0.5 * ((0.6 - 0.5 * 0.6) ** 2 + (-0.2 - 0.5 * 1.2 - 1) ** 2)})
wb.record("18.2f", 0.5 * ((0.6 + 0.5 * 0.6) ** 2 + (-0.2 - 0.5 * 1.2 - 1) ** 2), decimals=3, mistakes={"n'oublie pas le facteur ½": 4.05, "relis la question : on prend δ′ = y − ŷ": 0.225})
wb.record("18.2g", 2 * 0.6, decimals=2, mistakes={"sans le ½, la dérivée de (ŷ − y)² double": 0.6, "δ = ŷ − y, pas y − ŷ": -1.2})

WB_ANSWER {"decimals": 2, "hash": "640920bd5e1cd91991b4ca8a339efea8fd243e8ce62578d772419b4d2cd0bef2", "hash_coarse": "6fd30f163688c3dcaa01c38f12289919280f1d93d213074556627d1b5fc43281", "id": "18.2a", "kind": "float", "magnitude": -1, "mistakes": {"84fa2110b96025a34f9807e0f965a454bc45f7fae02ea401c61b725c8f68dfc0": "relis le label : y = (0 ; 1)", "ce043f6f2f45eae3a64c4cdfc3a82c381837856ba94527a6f7b926b34667dd3a": "n'oublie pas le facteur ½"}, "sign": 1}
📝 Ex 18.2a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "a71e61df7973740f995b26b0ec794bbf3e19ec3111bd943dc0e95ace6e344a0e", "hash_coarse": "d183a2227d61f2c455da2c36a31ea6aa6080bde74c9935d178d7fba12eb90e3b", "id": "18.2b", "kind": "float", "magnitude": -1, "mistakes": {"4addda7bbcdc6f223ee3636a17ab17f11490bc6ce90ba65d2d12596c29a1b1ed": "avec le ½, la dérivée de ½(ŷ − y)² est ŷ − y", "5615fe40f7d3d82d52101e410df8f2139a8cf2270fa2d27f395a9ed97433f43e": "δ = ŷ − y, pas y − ŷ"}, "sign": 1}
📝 Ex 18.2b : réponse

**Ex 18.3 — Descente sur une parabole : quel learning rate ?**

In [20]:
wb.record("18.3a", 1 - 0.1 * A_183, decimals=2, mistakes={"c'est ηa ; le facteur du pas est 1 − ηa": 0.4, "la courbure a = 4 intervient dans le facteur": 0.9})
wb.record("18.3b", w_183(0.1, 1), decimals=2, mistakes={"c'est l'écart w₁ − w* ; on demande w₁": 1.2, "le facteur multiplie l'écart au minimum, pas w₀": 1.8,
          "un pas : w₁ = w₀ − η L′(w₀), avec L′(w) = 4(w − 1)": 2.6})
wb.record("18.3c", w_183(0.1, 3), decimals=3, mistakes={"c'est l'écart w₃ − w* ; on demande w₃": 0.432, "le facteur multiplie l'écart au minimum, pas w₀": 0.648})
wb.record("18.3d", 1 / A_183, decimals=3, mistakes={"relis la question : on demande d'atteindre w* en un seul pas": 0.5,
          "un pas qui atteint w* annule le facteur 1 − ηa": 4.0, "le facteur 1 − ηa doit valoir 0": 1.0})
wb.record("18.3e", 2 / A_183, decimals=3, mistakes={"relis la question : on demande la limite au-delà de laquelle la descente diverge": 0.25,
          "la condition porte sur le facteur : |1 − ηa| < 1": 1.0, "le facteur doit rester au-dessus de −1": 0.125})
wb.record("18.3f", w_183(0.4, 2), decimals=2, mistakes={"relis la question : on demande w₂": -0.2, "le facteur 1 − ηa est négatif : son carré est positif": 0.28})
wb.record("18.3g", 2 / 0.1, decimals=1, mistakes={"la limite dépend de la courbure a = 0,1": 0.5, "relis la question : la limite de convergence, pas le pas qui atteint le minimum": 10.0,
          "a = 0,1 : refais la condition |1 − ηa| < 1": 2.0,
          "a est le double du coefficient : 0,05 (w − 1)² = (a/2)(w − 1)² avec a = 0,1": 40.0})
wb.record("18.3h", math.ceil(math.log(0.01 / 2) / math.log(0.6)), mistakes={"vérifie l'écart après le nombre de pas trouvé : il part de 2 et doit passer SOUS 0,01": 10,
          "l'écart de départ vaut |3 − 1| = 2, et le nombre de pas s'arrondit à l'entier supérieur": 9,
          "l'écart est multiplié par 0,6 à chaque pas, pas diminué de 0,6": 4},
          fractional="compte des pas entiers : le premier t pour lequel l'écart passe sous 0,01")

WB_ANSWER {"decimals": 2, "hash": "2c18f91a9d8a9c9b0b77018509e4ccd84cfab9782177823c7ceb280755dddc46", "hash_coarse": "d2b0d53c9402046fc8fd7c9f75566a1a2825ecb0668ca930aba4760b3c62b960", "id": "18.3a", "kind": "float", "magnitude": -1, "mistakes": {"71902e7270d598943818cb89a03cc95c61db0fabc839b66566dd143dd07b62bc": "la courbure a = 4 intervient dans le facteur", "ee74857ed98f9a1b9178d6682f07200f69102303e858c4474866f915c0432a60": "c'est ηa ; le facteur du pas est 1 − ηa"}, "sign": 1}
📝 Ex 18.3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "a3acfd850320da35c521a4136514ee0f3b4cfaa9382d5f47e70535999e8bf266", "hash_coarse": "f16e5b8c18827c06aa3da5292f9ca14e2d84304a795b11ba427212572cccaa9c", "id": "18.3b", "kind": "float", "magnitude": 0, "mistakes": {"42b72d3f54f09ab84cf85b49e74113f77702f30c8b6c7d431bae4f8a1e0aa95c": "c'est l'écart w₁ − w* ; on demande w₁", "91f42f7c6fde58366e0a78afc575aa718647a5d5c48d9da3840467f387a246e4": "un pas : w₁ = w₀ − η L′(w₀), avec

**Ex 18.4 — Rétropropagation complète du réseau minuscule, sans activation**

In [21]:
wb.record("18.4a", LIN_184['z1'].tolist(), decimals=3, mistakes={"une colonne de W₁ par neurone caché : A reçoit XA et YA": [2.5, 0.0]})
wb.record("18.4b", LIN_184['out'].tolist(), decimals=3, mistakes={"une colonne de W₂ par neurone de sortie : C reçoit AC et BC": [-2.0, 0.5]})
wb.record("18.4c", LIN_184['loss'], decimals=4, mistakes={"n'oublie pas le facteur ½": 2.125})
wb.record("18.4d", LIN_184['d2'].tolist(), decimals=3, mistakes={"δ = ŷ − y, pas y − ŷ": [0.75, -1.25]})
wb.record("18.4e", LIN_184['d1'].tolist(), decimals=3, mistakes={"le delta de A suit les fils qui PARTENT de A : AC vers C et AD vers D": [-0.125, 1.375],
          "il manque le chemin par D": [-0.75, -0.375], "δ = ŷ − y en sortie, pas y − ŷ": [2.0, -0.25]})
wb.record("18.4f", LIN_184['g2'].ravel().tolist(), decimals=3, mistakes={"le gradient d'un poids multiplie la SORTIE du neurone de départ par le delta d'arrivée": [-0.75, -1.25, -0.375, 0.625],
          "l'ordre est celui de W₂ ligne par ligne : AC, AD, BC, BD": [0.375, -1.125, -0.625, 1.875]})
wb.record("18.4g", LIN_184['g1'].ravel().tolist(), decimals=3, mistakes={"un poids de la première couche reçoit le delta du neurone CACHÉ qu'il alimente": [-0.75, 1.25, -1.5, 2.5],
          "l'ordre est celui de W₁ ligne par ligne : XA, XB, YA, YB": [-2.0, -4.0, 0.25, 0.5]})
wb.record("18.4h", -0.1 * GRAD2_184, decimals=4, mistakes={"multiplie par η = 0,1": -25.625, "les gradients se mettent au carré": 0.475,
          "le pas retire η g à chaque poids : la loss baisse": 2.5625})
wb.record("18.4i", tiny_184(W1_184 - 0.1 * LIN_184['g1'], W2_184 - 0.1 * LIN_184['g2'])['loss'], decimals=4, mistakes={"on retire η × gradient : w ← w − η g": tiny_184(W1_184 + 0.1 * LIN_184['g1'], W2_184 + 0.1 * LIN_184['g2'])['loss'],
          "la prévision h) ne vaut que pour de petits pas : refais la passe avant avec les nouveaux poids": 1.0625 - 2.5625})

WB_ANSWER {"decimals": 3, "element_coarse": [["2153dea31a751718cfd5a488e26267269637701920eb396818abc4f209511002"], ["a27ea159c95b4dadadab1da087454fc24d10a4113cd0d83a931137d372d2ed30"]], "element_hashes": ["cd0e741282477e3c87c24c7d79b10c95ae3e3f25252be1277ce33ac4f8887d4f", "025e9f900f26f9aeb3f3137d1ca69ce9a65a8bda70d06934f3b7f1a71d2dbdff"], "hash": "b46893c5c8ef6d4aec9e2c67c7bffc84257ad211b7c0521872f6d67795d2ba09", "id": "18.4a", "kind": "array", "mistakes": {"75334adb4d71bc659c232de0988d83d0fab880c6ecc3b170de42911da5d9666d": "une colonne de W₁ par neurone caché : A reçoit XA et YA"}, "shape": [2]}
📝 Ex 18.4a : réponse enregistrée (array, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "element_coarse": [["452b1b4c5486ed6f2d8d03c39bc8a28f3f06db7fceffaf7ed5e59d8c0ef5591a"], ["29d0f42c090c80eca7c0790ce726697bd6e096c825984cf7973b893121331907"]], "element_hashes": ["3cc5677a2c5f107b2c67b10d34a457dc1067d2112f5cceb13869f507dc370dbd", "7f661f11a46df34d7a610df9a011a74eb59ba8236e29ff6f6aae0fb74796cd7a

**Ex 18.5 — Le même réseau avec une sigmoïde cachée : le facteur σ'(z)**

In [22]:
wb.record("18.5a", SIG_185['a1'][0], decimals=4, mistakes={"c'est z_A ; applique la sigmoïde": -0.5, "relis la question : on demande la sortie σ(z_A), pas sa dérivée": SIG_185['a1'][0] * (1 - SIG_185['a1'][0])})
wb.record("18.5b", SIG_185['a1'][1], decimals=4, mistakes={"c'est z_B ; applique la sigmoïde": 1.5})
wb.record("18.5c", SIG_185['out'][0], decimals=4, mistakes={"relis l'énoncé : les neurones cachés ont maintenant une sigmoïde": 0.25,
          "les sorties restent sans activation : C_o = A_o AC + B_o BC": float(sigma(SIG_185['out'][0]))})
wb.record("18.5d", SIG_185['d2'][0], decimals=4, mistakes={"δ = ŷ − y, pas y − ŷ": -SIG_185['d2'][0], "la sortie C n'a pas d'activation : pas de facteur σ′ en sortie": SIG_185['d2'][0] * float(sigma(SIG_185['out'][0]) * (1 - sigma(SIG_185['out'][0]))),
          "tu as gardé la sortie C_o du réseau sans activation : avec la sigmoïde cachée, A_o et B_o changent, donc C_o aussi": LIN_184['d2'][0]})
wb.record("18.5e", float(sigma(-0.5) * (1 - sigma(-0.5))), decimals=4, mistakes={"σ′ se prend en z_A (ou σ(1 − σ) avec la sortie), pas en A_o": float(sigma(SIG_185['a1'][0]) * (1 - sigma(SIG_185['a1'][0]))),
          "σ′(z) = σ(z)(1 − σ(z)), pas σ(z)": float(sigma(-0.5))})
wb.record("18.5f", SIG_185['d1'][0], decimals=4, mistakes={"il manque le facteur σ′(z_A)": float(W2_184[0] @ SIG_185['d2']),
          "les deltas de sortie ne sont plus ceux de l'exercice 18.4 : refais la passe avant avec la sigmoïde": float(LIN_184['d1'][0] * sigma(-0.5) * (1 - sigma(-0.5))),
          "le facteur σ′ se prend en z_A, pas en A_o": float(W2_184[0] @ SIG_185['d2']) * float(sigma(SIG_185['a1'][0]) * (1 - sigma(SIG_185['a1'][0])))})
wb.record("18.5g", SIG_185['d1'][1], decimals=4, mistakes={"il manque le facteur σ′(z_B)": float(W2_184[1] @ SIG_185['d2'])})
wb.record("18.5h", SIG_185['g1'][1, 0], decimals=4, mistakes={"le poids YA part de Y : quelle entrée multiplie le delta ?": SIG_185['g1'][0, 0],
          "le poids YA arrive dans A : quel delta ?": SIG_185['g1'][1, 1]})

WB_ANSWER {"decimals": 4, "hash": "3c7c471a08a9b333864a987be29c29f64cf5301d57f6af4b45328af2bbf60a43", "hash_coarse": "a80b85849737c92a52462830da9bbb77aa552560b9e452eabcb822e5f0d31e91", "id": "18.5a", "kind": "float", "magnitude": -1, "mistakes": {"46acc9cea2b8aeaa293b0fe2a11a01d7c141fa62efa9e52c23bd8bddb1e6c60e": "relis la question : on demande la sortie σ(z_A), pas sa dérivée", "f7359310ba1339e0337572fc1537868b6915ccf6d20b7264e5400e24e49f5dc7": "c'est z_A ; applique la sigmoïde"}, "sign": 1}
📝 Ex 18.5a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "32147826511a728aec1ce4ea9b05e0c5e13239cc688180342714113e4aa17c14", "hash_coarse": "dffaa16cb57b7ef730ae8adccac95172b4cb1511ec745d759cfb6311f6618a08", "id": "18.5b", "kind": "float", "magnitude": -1, "mistakes": {"27fd81674a35535e24effb1ff361f4b2b2a2be04b69f9e88d26a20e721e97e52": "c'est z_B ; applique la sigmoïde"}, "sign": 1}
📝 Ex 18.5b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4,

**Ex 18.9 — Fermi : rétropropagation contre force brute**

In [23]:
wb.record("18.9a", P_189, mistakes={"les biais sont des paramètres aussi": F_189, "compte aussi la dernière couche, 128 × 10 + 10": P_189 - 1290})
wb.record("18.9b", round(F_189 / 1000), mistakes={"en milliers : divise par 1 000": F_189, "une multiplication par poids, pour UNE image": round(F_189 * 60000 / 1000)})
wb.record("18.9c", 2 * P_189, mistakes={"en différence centrée, deux passes par paramètre": P_189, "deux passes par paramètre, biais compris": 2 * F_189})
wb.record("18.9d", round(2 * P_189 / 3 / 1000), mistakes={"la rétropropagation complète coûte une passe avant ET une passe arrière (deux passes avant)": round(2 * P_189 / 1000),
          "une passe arrière vaut deux passes avant, et il faut aussi la passe avant": round(2 * P_189 / 2 / 1000),
          "en milliers : divise par 1 000": round(2 * P_189 / 3)})
wb.record("18.9e", round(3 * F_189 * 60000 / 1e12 * 1000), mistakes={"compte la passe avant ET la passe arrière (deux passes avant)": round(F_189 * 60000 / 1e12 * 1000),
          "la passe arrière vaut deux passes avant : trois en tout": round(2 * F_189 * 60000 / 1e12 * 1000),
          "en millisecondes : 1 s = 1 000 ms": round(3 * F_189 * 60000 / 1e12 * 1e6)})
wb.record("18.9f", 2 * P_189 * F_189 * 60000 / 1e12 / 3600, decimals=1, mistakes={"en différence centrée, deux passes avant par paramètre": P_189 * F_189 * 60000 / 1e12 / 3600,
          "en heures : 1 h = 3 600 s": 2 * P_189 * F_189 * 60000 / 1e12 / 60})

WB_ANSWER {"hash": "51bcb45618ede13a543cf0b28cb824176aa28e303a33c9b040ce9749a02a035b", "id": "18.9a", "kind": "int", "magnitude": 5, "mistakes": {"a9a8f6d09ff0217499bc99afe171343ee5d43af50b422920762a959be2a5e7e7": "les biais sont des paramètres aussi", "e4b846dcc114ca43587816a1808fc2656ac01ff2363916b8edaac4c299baf486": "compte aussi la dernière couche, 128 × 10 + 10"}, "sign": 1}
📝 Ex 18.9a : réponse enregistrée (int).
WB_ANSWER {"hash": "95bcfcf9448a8e200a44aa93f51346e5180ebaa97670a0dea052e705e0dca409", "id": "18.9b", "kind": "int", "magnitude": 2, "mistakes": {"62ba180c5dea2b95ee3c2d76a035630b99699c10cc694f836d917fece813e3f1": "en milliers : divise par 1 000", "a0251129c207aa0c58e47f54f2ad83c2d6a9a74bdb9a6b7c4fb1cf88a5c70009": "une multiplication par poids, pour UNE image"}, "sign": 1}
📝 Ex 18.9b : réponse enregistrée (int).
WB_ANSWER {"hash": "4fa0dcaac70a442e4a62c9b7cae7ef120df52cef5c30ff60eecf249c3d4a92de", "id": "18.9c", "kind": "int", "magnitude": 5, "mistakes": {"3db08cc3ee5aba

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dérouler à la main la passe avant, les deltas et un pas de mise à jour d'un petit réseau ?
2. Sais-tu écrire les gradients d'une couche dense pour un mini-batch, et vérifier leurs formes ?
3. Sais-tu vérifier un gradient par différences finies, et reconnaître un learning rate mal réglé ?

**Pour aller plus loin** : le chapitre 2 du livre de M. Nielsen, les notes de CS231n et micrograd d'A. Karpathy, cités dans la fiche. La suite : le ch. 19, où les optimiseurs remplaceront la mise à jour $w \leftarrow w - \eta g$.

*Les exercices de code 18.12 à 18.29 (parties A à D, ta librairie `nn/backward.py` et `autograd_mini.py`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 18` pour obtenir le notebook complet.*